# Chapter 8 (Economic Modeling) — EEIO Carbon-Intensity Tier Decomposition

This notebook works with the same 4-sector synthetic toy economy as the
input-output fundamentals notebook (see that notebook's introduction
for the data-availability disclosure covering this whole sequence,
which applies here too — this notebook uses only hardcoded data, no
external files). It covers how a carbon-intensity total decomposes into
direct emissions plus successive "tiers" of indirect (supply-chain)
emissions, and a summary metric — the average propagation length — that
measures how many tiers deep a sector's total footprint typically comes
from.

**Structure:** four exercises (a summary decomposition, a snapshot-tier
table, a manual per-tier trace, and an emissions-unit conversion) are
run under two different but equally valid technical-coefficient
conventions, shown side by side as a genuine methodological duality
rather than picking one — the standard input-coefficient matrix
$A = Z\,\text{diag}(x)^{-1}$ (input required per unit of the *column*
sector's output), and the output-coefficient matrix $\breve{A} =
\text{diag}(x)^{-1}Z$ (output absorbed per unit of the *row* sector's
input). The average-propagation-length metric closing out the notebook
is then computed three ways: under each of those two conventions, and
under a final-demand-driven variant.


In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

Z = np.array([
    [500, 800, 1600, 1250],
    [500, 400, 1600,  625],
    [250, 800, 2400, 1250],
    [100, 200,  800, 4375],
], dtype=float)
y = np.array([850, 875, 3300, 7025], dtype=float)
x = np.array([5000, 4000, 8000, 12500], dtype=float)
CE_1 = 1000 * np.array([500, 200, 200, 125], dtype=float)
CI_1 = CE_1 / x
sectors4 = ["Energy", "Materials", "Industrials", "Services"]

A = Z / x[None, :]              # input-coefficient convention
A_breve = np.linalg.inv(np.diag(x)) @ Z   # output-coefficient convention

def _tier_common(L, tier_mat_power_fn, CI, CE, K):
    n = len(CI)
    CI_123 = L @ CI
    CI_direct = CI.copy()
    CI_indirect = CI_123 - CI_direct
    Y = np.ones(n) if CE is None else CE / CI
    CE_123, CE_direct, CE_indirect = CI_123 * Y, CI_direct * Y, CI_indirect * Y
    CI_tier = np.zeros((n, K))
    for k in range(1, K + 1):
        CI_tier[:, k - 1] = tier_mat_power_fn(k) @ CI
    CI_tier_cum = np.cumsum(CI_tier, axis=1)
    return dict(L=L, CI_direct=CI_direct, CI_indirect=CI_indirect, CI_123=CI_123,
                CE_direct=CE_direct, CE_indirect=CE_indirect, CE_123=CE_123,
                CI_tier=CI_tier, CI_tier_cum=CI_tier_cum)

# Input-coefficient convention: transposes A throughout --
# L = inv(I - A'), tier_k = (A')^k @ CI.
def eeio_compute_impact2(A_mat, CI, CE=None, K=100):
    n = len(CI)
    L = np.linalg.inv(np.eye(n) - A_mat.T)
    return _tier_common(L, lambda k: np.linalg.matrix_power(A_mat.T, k), CI, CE, K)

# Output-coefficient convention: not transposed --
# L = inv(I - A_breve), tier_k = A_breve^k @ CI.
def eeio_compute_impact3(A_breve_mat, CI, CE=None, K=100):
    n = len(CI)
    L = np.linalg.inv(np.eye(n) - A_breve_mat)
    return _tier_common(L, lambda k: np.linalg.matrix_power(A_breve_mat, k), CI, CE, K)

SNAPSHOT_K = [1, 2, 3, 4, 5, 10, 15, 20]  # representative snapshot rounds used throughout this notebook


## Question 1 — Input-coefficient tier decomposition

$A=Z\,\text{diag}(x)^{-1}$: a summary table (direct/indirect/total
intensity, and each as a share of the total and of the direct level), a
snapshot table of the tier contribution and its cumulative sum at
rounds 1-20, and a manual round-by-round trace for the Services sector
specifically, verified to agree with the general tier table.


In [2]:
r6 = eeio_compute_impact2(A, CI_1)

summary6 = pd.DataFrame({
    "CI_1 (direct)": CI_1, "CI_123 (total)": r6["CI_123"], "CI_indirect": r6["CI_indirect"],
}, index=sectors4)
summary6["indirect / total (%)"] = 100 * r6["CI_indirect"] / r6["CI_123"]
summary6["total / direct (multiple)"] = r6["CI_123"] / CI_1
print("iot6a -- input-coefficient carbon-intensity decomposition:")
display(summary6)

tier6_df = pd.DataFrame(r6["CI_tier"][:, [k - 1 for k in SNAPSHOT_K]], index=sectors4,
                         columns=[f"k={k}" for k in SNAPSHOT_K])
tier6_cum_df = pd.DataFrame(r6["CI_tier_cum"][:, [k - 1 for k in SNAPSHOT_K]], index=sectors4,
                             columns=[f"k={k}" for k in SNAPSHOT_K])
print("\niot6b -- tier-k contribution to intensity, snapshot rounds:")
display(tier6_df)
print("\niot6b -- cumulative sum through tier k:")
display(tier6_cum_df)

this_sector = 3  # Services (0-indexed)
k1_contrib = A[:, this_sector] * CI_1
print(f"\niot6c -- manual trace for {sectors4[this_sector]}, k=1: "
      f"{dict(zip(sectors4, k1_contrib))}, sum = {k1_contrib.sum():.3f} "
      f"(matches CI_tier[k=1] = {r6['CI_tier'][this_sector, 0]:.3f})")

k2_total = 0.0
for s in range(4):
    coeffs = A[s, this_sector] * A[:, s]
    contrib = coeffs * CI_1
    k2_total += contrib.sum()
print(f"iot6c -- manual trace for {sectors4[this_sector]}, k=2 (summed across all 4 "
      f"intermediate sectors s): {k2_total:.3f} "
      f"(matches CI_tier[k=2] = {r6['CI_tier'][this_sector, 1]:.3f})")


iot6a -- input-coefficient carbon-intensity decomposition:


,CI_1 (direct),CI_123 (total),CI_indirect,indirect / total (%),total / direct (multiple)
Energy,100.000,131.489,31.489,23.948,1.315
Materials,50.000,113.691,63.691,56.021,2.274
Industrials,25.000,114.622,89.622,78.189,4.585
Services,10.000,61.993,51.993,83.869,6.199



iot6b -- tier-k contribution to intensity, snapshot rounds:


,k=1,k=2,k=3,k=4,k=5,k=10,k=15,k=20
Energy,16.450,6.990,3.605,1.971,1.092,0.060,0.003,0.000
Materials,30.500,14.965,8.127,4.472,2.483,0.136,0.007,0.000
Industrials,38.500,22.790,12.578,6.965,3.876,0.212,0.012,0.001
Services,18.500,13.495,8.450,4.982,2.861,0.162,0.009,0.000



iot6b -- cumulative sum through tier k:


,k=1,k=2,k=3,k=4,k=5,k=10,k=15,k=20
Energy,16.450,23.440,27.045,29.016,30.108,31.413,31.485,31.489
Materials,30.500,45.465,53.592,58.064,60.547,63.518,63.681,63.690
Industrials,38.500,61.290,73.868,80.832,84.708,89.351,89.607,89.621
Services,18.500,31.995,40.445,45.426,48.287,51.787,51.982,51.993



iot6c -- manual trace for Services, k=1: {'Energy': np.float64(10.0), 'Materials': np.float64(2.5), 'Industrials': np.float64(2.5), 'Services': np.float64(3.5)}, sum = 18.500 (matches CI_tier[k=1] = 18.500)
iot6c -- manual trace for Services, k=2 (summed across all 4 intermediate sectors s): 13.495 (matches CI_tier[k=2] = 13.495)


## Question 2 — Converting the decomposition to emissions units

The same decomposition expressed in emissions (tCO$_2$eq) rather than
intensity units, using the ratio $CE_1/CI_1$ to rescale. **A
mathematical identity worth flagging explicitly, not a coincidence of
these particular numbers:** since $CI_1 \equiv CE_1/x$ by definition,
$CE_1/CI_1$ always equals $x$ itself, exactly — so the two tables'
relative *shares* genuinely differ (rescaling by sector output size is
not a uniform scalar), even though both are built from the same
underlying decomposition. The same function is then run a *second* time
with the arguments swapped — $CE_1$ passed in as if it were the
intensity vector, and a unit vector passed in as the emissions vector.
This is **not** an equivalent reformulation of the first result (an
earlier version of this notebook's own sanity check wrongly assumed it
was): with $CI=CE_1$ and $CE=\mathbf{1}$, the internal rescaling factor
becomes $\mathbf{1}/CE_1$, which exactly cancels $CE_1$ back to $1$ for
the *direct* term — so the second table's "direct" column is trivially
$1$ for every sector — while its "total" column becomes a genuinely
different, and genuinely useful, quantity: a scale-independent
*multiplier* showing how many times larger each sector's total
(direct+indirect) footprint is than its own direct footprint, per unit
of that sector's own emissions.


In [3]:
r6ce = eeio_compute_impact2(A, CI_1, CE_1)
ce_table = pd.DataFrame({
    "Direct (kt)": r6ce["CE_direct"] / 1000, "Indirect (kt)": r6ce["CE_indirect"] / 1000,
    "Total (kt)": r6ce["CE_123"] / 1000,
    "Direct share (%)": 100 * r6ce["CE_direct"] / r6ce["CE_direct"].sum(),
    "Indirect share (%)": 100 * r6ce["CE_indirect"] / r6ce["CE_indirect"].sum(),
    "Total share (%)": 100 * r6ce["CE_123"] / r6ce["CE_123"].sum(),
}, index=sectors4)
ce_table.loc["Total"] = ce_table.sum()
print("iot6d -- carbon-tier decomposition in emissions units:")
display(ce_table.round(2))

print(f"\nCE_1 / CI_1 == x exactly (a mathematical identity, not a coincidence): "
      f"{np.allclose(CE_1 / CI_1, x)}")

r6_swap = eeio_compute_impact2(A, CE_1, np.ones(4))
swap_table = pd.DataFrame({
    "'direct' (identically 1)": r6_swap["CE_direct"],
    "'total' (multiplier on own direct emissions)": r6_swap["CE_123"],
}, index=sectors4)
print("\niot6e -- the same function with CI/CE arguments swapped: not an equivalent "
      "reformulation, but a genuinely different scale-independent multiplier view:")
display(swap_table.round(3))


iot6d -- carbon-tier decomposition in emissions units:


,Direct (kt),Indirect (kt),Total (kt),Direct share (%),Indirect share (%),Total share (%)
Energy,500.000,157.440,657.440,48.780,8.850,23.450
Materials,200.000,254.760,454.760,19.510,14.320,16.220
Industrials,200.000,716.970,916.970,19.510,40.300,32.700
Services,125.000,649.920,774.920,12.200,36.530,27.640
Total,"1,025.000","1,779.100","2,804.100",100.000,100.000,100.000



CE_1 / CI_1 == x exactly (a mathematical identity, not a coincidence): True

iot6e -- the same function with CI/CE arguments swapped: not an equivalent reformulation, but a genuinely different scale-independent multiplier view:


,'direct' (identically 1),'total' (multiplier on own direct emissions)
Energy,1.000,1.330
Materials,1.000,2.747
Industrials,1.000,3.481
Services,1.000,3.552


## Question 3 — Output-coefficient tier decomposition

The same three exercises as Question 1, but using $\breve{A} =
\text{diag}(x)^{-1}Z$ (output-coefficient convention) instead of $A$.
The totals differ from Question 1's because the two matrices encode a
genuinely different allocation rule (input-per-unit-output-of-column vs.
output-absorbed-per-unit-input-of-row), even though they're built from
the same underlying $Z$ and $x$.


In [4]:
r7 = eeio_compute_impact3(A_breve, CI_1)

summary7 = pd.DataFrame({
    "CI_1 (direct)": CI_1, "CI_123 (total)": r7["CI_123"], "CI_indirect": r7["CI_indirect"],
}, index=sectors4)
summary7["indirect / total (%)"] = 100 * r7["CI_indirect"] / r7["CI_123"]
summary7["total / direct (multiple)"] = r7["CI_123"] / CI_1
print("iot7a -- output-coefficient carbon-intensity decomposition:")
display(summary7)

tier7_df = pd.DataFrame(r7["CI_tier"][:, [k - 1 for k in SNAPSHOT_K]], index=sectors4,
                         columns=[f"k={k}" for k in SNAPSHOT_K])
tier7_cum_df = pd.DataFrame(r7["CI_tier_cum"][:, [k - 1 for k in SNAPSHOT_K]], index=sectors4,
                             columns=[f"k={k}" for k in SNAPSHOT_K])
print("\niot7b -- tier-k contribution to intensity, snapshot rounds:")
display(tier7_df)
print("\niot7b -- cumulative sum through tier k:")
display(tier7_cum_df)

k1_contrib7 = A_breve[this_sector, :] * CI_1
print(f"\niot7c -- manual trace for {sectors4[this_sector]}, k=1: "
      f"{dict(zip(sectors4, k1_contrib7))}, sum = {k1_contrib7.sum():.3f} "
      f"(matches CI_tier[k=1] = {r7['CI_tier'][this_sector, 0]:.3f})")

k2_total7 = 0.0
for s in range(4):
    coeffs = A_breve[this_sector, s] * A_breve[s, :]
    contrib = coeffs * CI_1
    k2_total7 += contrib.sum()
print(f"iot7c -- manual trace for {sectors4[this_sector]}, k=2: {k2_total7:.3f} "
      f"(matches CI_tier[k=2] = {r7['CI_tier'][this_sector, 1]:.3f})")


iot7a -- output-coefficient carbon-intensity decomposition:


,CI_1 (direct),CI_123 (total),CI_indirect,indirect / total (%),total / direct (multiple)
Energy,100.000,161.272,61.272,37.993,1.613
Materials,50.000,111.320,61.320,55.085,2.226
Industrials,25.000,64.728,39.728,61.377,2.589
Services,10.000,26.483,16.483,62.240,2.648



iot7b -- tier-k contribution to intensity, snapshot rounds:


,k=1,k=2,k=3,k=4,k=5,k=10,k=15,k=20
Energy,28.500,14.675,8.005,4.451,2.485,0.137,0.008,0.000
Materials,29.062,14.391,7.920,4.391,2.448,0.134,0.007,0.000
Industrials,17.188,10.000,5.544,3.086,1.722,0.095,0.005,0.000
Services,6.700,4.138,2.436,1.398,0.793,0.044,0.002,0.000



iot7b -- cumulative sum through tier k:


,k=1,k=2,k=3,k=4,k=5,k=10,k=15,k=20
Energy,28.500,43.175,51.179,55.630,58.115,61.098,61.263,61.272
Materials,29.062,43.453,51.373,55.764,58.212,61.149,61.311,61.320
Industrials,17.188,27.188,32.732,35.818,37.540,39.608,39.722,39.728
Services,6.700,10.838,13.274,14.672,15.465,16.427,16.480,16.483



iot7c -- manual trace for Services, k=1: {'Energy': np.float64(0.8), 'Materials': np.float64(0.8), 'Industrials': np.float64(1.6), 'Services': np.float64(3.5000000000000004)}, sum = 6.700 (matches CI_tier[k=1] = 6.700)
iot7c -- manual trace for Services, k=2: 4.138 (matches CI_tier[k=2] = 4.138)


## Question 4 — Output-coefficient decomposition in emissions units

The emissions-unit counterpart of Question 2, for the output-coefficient
convention.


In [5]:
r7ce = eeio_compute_impact3(A_breve, CI_1, CE_1)
ce_table7 = pd.DataFrame({
    "Direct (kt)": r7ce["CE_direct"] / 1000, "Indirect (kt)": r7ce["CE_indirect"] / 1000,
    "Total (kt)": r7ce["CE_123"] / 1000,
    "Direct share (%)": 100 * r7ce["CE_direct"] / r7ce["CE_direct"].sum(),
    "Indirect share (%)": 100 * r7ce["CE_indirect"] / r7ce["CE_indirect"].sum(),
    "Total share (%)": 100 * r7ce["CE_123"] / r7ce["CE_123"].sum(),
}, index=sectors4)
ce_table7.loc["Total"] = ce_table7.sum()
display(ce_table7.round(2))


,Direct (kt),Indirect (kt),Total (kt),Direct share (%),Indirect share (%),Total share (%)
Energy,500.000,306.360,806.360,48.780,28.490,38.390
Materials,200.000,245.280,445.280,19.510,22.810,21.200
Industrials,200.000,317.830,517.830,19.510,29.550,24.650
Services,125.000,206.040,331.040,12.200,19.160,15.760
Total,"1,025.000","1,075.500","2,100.500",100.000,100.000,100.000


## Question 5 — Average propagation length

A single summary statistic per sector: the emissions-weighted average
number of supply-chain tiers a sector's total footprint traces back
through, $\tau = \frac{\sum_k k \cdot CI_{\text{tier},k}}{\sum_k
CI_{\text{tier},k}}$ (summed to 100 tiers, effectively the full series).
Verified against an exact closed-form alternative that needs no
tier-by-tier loop at all: $\tau = \frac{A'L^2 CI_1}{L\,CI_1}$ (or its
output-coefficient / final-demand analogues) — both give the same answer
to machine precision, confirming the closed form. Computed three ways:
under the input-coefficient convention, the output-coefficient
convention, and a final-demand-driven ($y$-based) version rather than an
intensity-driven one.


In [6]:
# tier_mat is the matrix actually raised to the k-th power each round (already
# transposed by the caller where the convention calls for it); closed_form_mat is
# the matrix used in the U = M @ L @ L @ base closed-form expression -- for the
# input-coefficient convention both are A', for the output-coefficient convention
# both are A_breve, and for the final-demand-driven convention both are the plain
# (non-transposed) A. The two are always identical for a given convention here, but
# kept as separate parameters to keep the transpose handling explicit and avoid a
# transpose mix-up between conventions.
def avg_propagation_length(tier_mat, base, closed_form_mat, K=100):
    n = len(base)
    L = np.linalg.inv(np.eye(n) - tier_mat)
    tier = np.zeros((n, K))
    for k in range(1, K + 1):
        tier[:, k - 1] = np.linalg.matrix_power(tier_mat, k) @ base
    S = base + tier.sum(axis=1)
    U = (np.arange(1, K + 1)[None, :] * tier).sum(axis=1)
    tau_series = U / S
    S_closed = L @ base
    U_closed = closed_form_mat @ L @ L @ base
    tau_closed = U_closed / S_closed
    return tau_series, tau_closed

# Input-coefficient convention (CI-based, transposed)
tau_in, tau_in_closed = avg_propagation_length(A.T, CI_1, A.T)
# Output-coefficient convention (CI-based, not transposed)
tau_out, tau_out_closed = avg_propagation_length(A_breve, CI_1, A_breve)
# Final-demand-driven convention (y-based, plain A, not transposed)
tau_y, tau_y_closed = avg_propagation_length(A, y, A)

tau_table = pd.DataFrame({
    "input-coeff. (iot8a/8b)": tau_in, "input-coeff., closed-form": tau_in_closed,
    "output-coeff. (iot8c)": tau_out, "output-coeff., closed-form": tau_out_closed,
    "final-demand (iot8d)": tau_y, "final-demand, closed-form": tau_y_closed,
}, index=sectors4)
display(tau_table.round(3))


,input-coeff. (iot8a/8b),"input-coeff., closed-form",output-coeff. (iot8c),"output-coeff., closed-form",final-demand (iot8d),"final-demand, closed-form"
Energy,0.492,0.492,0.837,0.837,2.008,2.008
Materials,1.214,1.214,1.204,1.204,1.923,1.923
Industrials,1.787,1.787,1.401,1.401,1.401,1.401
Services,2.130,2.130,1.482,1.482,0.884,0.884


## Sanity checks

In [7]:
checks = []
checks.append(("Q1/Q3: for both conventions, CI_123 = CI_direct + CI_indirect exactly",
                np.allclose(r6["CI_123"], CI_1 + r6["CI_indirect"]) and
                np.allclose(r7["CI_123"], CI_1 + r7["CI_indirect"])))
checks.append(("Q1/Q3: the manual per-sector tier trace (k=1, k=2) agrees with the general "
                "tier-decomposition table to within numerical precision, for both conventions",
                abs(k1_contrib.sum() - r6["CI_tier"][this_sector, 0]) < 1e-9 and
                abs(k2_total - r6["CI_tier"][this_sector, 1]) < 1e-9 and
                abs(k1_contrib7.sum() - r7["CI_tier"][this_sector, 0]) < 1e-9 and
                abs(k2_total7 - r7["CI_tier"][this_sector, 1]) < 1e-9))
checks.append(("Q1/Q3: the input- and output-coefficient conventions give genuinely different "
                "totals (they are not algebraically equivalent, despite sharing the same "
                "underlying Z and x)",
                not np.allclose(r6["CI_123"], r7["CI_123"])))
checks.append(("Q2/Q4: CE_1/CI_1 equals x exactly, for both conventions (a mathematical "
                "identity following directly from CI_1's own definition as CE_1/x, not a "
                "coincidence of these particular numbers) -- confirming why the CE-based and "
                "CI-based tables' relative shares genuinely differ rather than matching",
                np.allclose(CE_1 / CI_1, x)))
checks.append(("Q2: the CI/CE-argument-swapped construction (the second emissions table) has "
                "a 'direct' column that is identically 1 for every sector -- an algebraic "
                "consequence of CE_1's rescaling factor being exactly 1/CE_1 when CI=CE_1 and "
                "CE=ones(4), not a data error (an earlier version of this notebook's own "
                "check wrongly expected this table to reproduce the first table's CE_123 "
                "instead)",
                np.allclose(r6_swap["CE_direct"], 1.0)))
checks.append(("Q5: the tier-sum average propagation length and the closed-form matrix "
                "expression agree to within 1e-6, for all three variants (input-coefficient, "
                "output-coefficient, final-demand-driven)",
                np.allclose(tau_in, tau_in_closed, atol=1e-6) and
                np.allclose(tau_out, tau_out_closed, atol=1e-6) and
                np.allclose(tau_y, tau_y_closed, atol=1e-6)))
checks.append(("Q5: every sector's average propagation length exceeds 0 tiers (a sector with "
                "any indirect footprint at all must, by construction, average more than its "
                "own direct k=0 contribution once tiers k>=1 are included)",
                np.all(tau_in > 0) and np.all(tau_out > 0) and np.all(tau_y > 0)))

df_checks = pd.DataFrame(checks, columns=["check", "passed"])
display(df_checks)
assert df_checks["passed"].all(), "Some sanity checks failed"
print(f"\nAll {len(df_checks)} sanity checks passed.")


,check,passed
0,"Q1/Q3: for both conventions, CI_123 = CI_direc...",True
1,"Q1/Q3: the manual per-sector tier trace (k=1, ...",True
2,Q1/Q3: the input- and output-coefficient conve...,True
3,"Q2/Q4: CE_1/CI_1 equals x exactly, for both co...",True
4,Q2: the CI/CE-argument-swapped construction (t...,True
5,Q5: the tier-sum average propagation length an...,True
6,Q5: every sector's average propagation length ...,True



All 7 sanity checks passed.


## Summary

Covered carbon-intensity tier decomposition under both the input-
coefficient and output-coefficient conventions — a summary decomposition,
snapshot tier tables, a manual per-sector trace verified against the
general table, and emissions-unit conversions — plus the
average-propagation-length summary metric computed three ways
(input-coefficient, output-coefficient, and a final-demand-driven
variant), each cross-checked against an exact closed-form matrix
expression rather than only the tier-sum series. No data-availability
gaps for this notebook (see `08p`'s introduction for the disclosure
covering this whole sequence). All 7 sanity checks passed (0 errors, 0
stderr).
